# M19 — Make Models Learn with Gradients

**Objective:** connect changing parameters to changing loss through gradients. We begin with one adjustable number and only introduce a second parameter after the scalar mechanism works.

## Working contract

Every experiment follows **predict → act → observe → explain**. Keep predictions in your own notes before running an action cell. The notebook is deterministic, CPU-only, offline, and uses no secrets or paid APIs.

In [ ]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M19" / "gradient_core.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run this notebook from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M19.gradient_core import (
    analytic_weight_gradient,
    finite_difference_gradient,
    linear_loss,
    one_parameter_loss,
    one_parameter_step,
    predict_one_parameter,
    run_linear_descent,
    run_one_parameter_descent,
)

np.set_printoptions(precision=4, suppress=True)

## Tiny observable system

The first target follows `y = 3x`, but treat that as a check—not as the procedure. Our first model has only one parameter: `prediction = weight * x`.

In [ ]:
data_path = ROOT / "datasets" / "M19" / "tiny_linear.csv"
fixture = np.genfromtxt(data_path, delimiter=",", names=True)
xs = fixture["x"].astype(float)
ys = fixture["y_one_parameter"].astype(float)
ys_with_bias = fixture["y_with_bias"].astype(float)
print("x             :", xs)
print("one parameter :", ys)
print("with bias     :", ys_with_bias)

## The observable chain

For now: choose one `weight` → multiply every `x` to get predictions → compare predictions with targets → average squared errors to get one loss. No derivative is needed to perform that chain.

### Predict before running — manual parameter changes

For weights `0, 1, 2, 3, 4`, predict which has the smallest loss. Choose one input and write how its prediction changes as weight increases. Do this before revealing any computed losses.

In [ ]:
candidate_weights = [0.0, 1.0, 2.0, 3.0, 4.0]
manual_results = []
for weight in candidate_weights:
    predictions = predict_one_parameter(xs, weight)
    loss = one_parameter_loss(xs, ys, weight)
    manual_results.append((weight, predictions, loss))
    print(f"weight={weight:.1f} predictions={np.array(predictions)} loss={loss:.2f}")

### Explain the observation

Which weights made loss fall, and when did it rise again? Explain this using prediction errors, not the phrase “closer to the answer.”

### Predict before running — loss curve

Sketch the curve you expect between weights `-1` and `5`. Mark the likely minimum and where you expect the slope to be negative, zero, and positive.

In [ ]:
curve_weights = np.linspace(-1.0, 5.0, 241)
curve_losses = np.array([one_parameter_loss(xs, ys, w) for w in curve_weights])
minimum_index = int(np.argmin(curve_losses))
print(f"sampled minimum: weight={curve_weights[minimum_index]:.3f}, loss={curve_losses[minimum_index]:.3f}")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve_weights, curve_losses, label="mean squared loss")
ax.scatter([row[0] for row in manual_results], [row[2] for row in manual_results], color="#d1495b", label="manual weights")
ax.set(xlabel="weight", ylabel="loss", title="One parameter changes one loss")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## Gradient intuition

At one point on the loss curve, the gradient is its local slope with respect to the parameter. A negative gradient says increasing the weight locally lowers loss; a positive gradient says increasing the weight locally raises loss. A larger magnitude means a steeper local change. The gradient is evidence about a neighborhood, not the location of the minimum by itself.

### Predict before running — finite difference

At `weight = 1.5`, predict the gradient sign from the curve. Then predict whether `loss(weight + epsilon)` or `loss(weight - epsilon)` will be larger.

In [ ]:
probe_weight = 1.5
epsilon = 1.0e-5
loss_minus = one_parameter_loss(xs, ys, probe_weight - epsilon)
loss_plus = one_parameter_loss(xs, ys, probe_weight + epsilon)
finite_gradient = finite_difference_gradient(xs, ys, probe_weight, epsilon)
print(f"loss(w-epsilon)={loss_minus:.8f}")
print(f"loss(w+epsilon)={loss_plus:.8f}")
print(f"central finite-difference gradient={finite_gradient:.8f}")

The central difference computes `(loss(w + epsilon) - loss(w - epsilon)) / (2 * epsilon)`. It turns two nearby loss measurements into a local slope estimate.

## A simple analytic derivative

For `prediction_i = weight * x_i` and mean squared loss `L = (1/n) Σ(prediction_i - y_i)²`, the chain rule gives:

`dL/dweight = (2/n) Σ x_i * (weight * x_i - y_i)`

The residual says how the prediction missed; multiplying by `x_i` says how sensitive that prediction is to the weight.

In [ ]:
analytic_gradient = analytic_weight_gradient(xs, ys, probe_weight)
gradient_gap = abs(finite_gradient - analytic_gradient)
print(f"finite difference: {finite_gradient:.8f}")
print(f"analytic        : {analytic_gradient:.8f}")
print(f"absolute gap    : {gradient_gap:.3e}")
assert gradient_gap < 1.0e-6

### Predict before running — one parameter update

Start at `weight = 1.0` with learning rate `0.2`. Calculate the analytic gradient and the proposed update `weight - learning_rate * gradient` by hand. Predict whether the next loss is above or below the current loss.

In [ ]:
correct_step = one_parameter_step(xs, ys, weight=1.0, learning_rate=0.2)
print("parameter before:", correct_step.parameters_before)
print("predictions     :", np.array(correct_step.predictions))
print("loss before    :", correct_step.loss_before)
print("gradient       :", correct_step.gradient)
print("parameter after:", correct_step.parameters_after)
print("loss after     :", correct_step.loss_after)
assert correct_step.loss_after < correct_step.loss_before

### Predict before running — multiple steps

Predict how the weight, gradient magnitude, and loss will change over eight correct steps. Will equal learning-rate steps produce equal parameter changes?

In [ ]:
scalar_trace = run_one_parameter_descent(
    xs, ys, initial_weight=1.0, learning_rate=0.2, steps=8
)
for record in scalar_trace:
    print(
        f"step={record.step:02d} "
        f"weight={record.parameters_before[0]:.6f} "
        f"gradient={record.gradient[0]:.6f} "
        f"next_weight={record.parameters_after[0]:.6f} "
        f"next_loss={record.loss_after:.8f}"
    )

In [ ]:
scalar_losses = [scalar_trace[0].loss_before] + [record.loss_after for record in scalar_trace]
assert all(after < before for before, after in zip(scalar_losses, scalar_losses[1:]))
assert abs(scalar_trace[-1].parameters_after[0] - 3.0) < 1.0e-5
print("Every scalar update lowered loss.")

## Controlled failure — wrong gradient sign

### Predict before running

Keep the same start, data, gradient, and learning rate, but replace subtraction with addition. Calculate the faulty next weight and predict whether its loss rises or falls.

In [ ]:
failure_start = 1.0
failure_rate = 0.2
failure_gradient = analytic_weight_gradient(xs, ys, failure_start)
faulty_weight = failure_start + failure_rate * failure_gradient  # deliberately wrong sign
failure_loss_before = one_parameter_loss(xs, ys, failure_start)
failure_loss_after = one_parameter_loss(xs, ys, faulty_weight)
print(f"start={failure_start:.2f}, gradient={failure_gradient:.2f}, faulty next={faulty_weight:.2f}")
print(f"loss: {failure_loss_before:.2f} -> {failure_loss_after:.2f}")
assert failure_loss_after > failure_loss_before

### Diagnose and repair

Trace the failure as `parameter → prediction → loss → gradient → update`. The data, predictions, loss, and gradient are unchanged; the smallest repair is the update operator. Explain why changing `+` back to `-` follows the local slope evidence.

In [ ]:
repaired = one_parameter_step(xs, ys, failure_start, failure_rate)
print(f"repaired next={repaired.parameters_after[0]:.2f}, loss={repaired.loss_after:.2f}")
assert repaired.loss_after < repaired.loss_before

## Only now: multiple parameters

Extend the model to `prediction = weight * x + bias` and use the offset target `3x + 2`. The gradient is now a pair: one partial derivative for weight and one for bias. Each component answers how loss changes when only its parameter changes locally.

### Predict before running — weight and bias

At `weight = 0, bias = 0`, predict the sign of each gradient component. Based on the target rule, predict the values repeated descent should approach.

In [ ]:
linear_trace = run_linear_descent(
    xs,
    ys_with_bias,
    initial_weight=0.0,
    initial_bias=0.0,
    learning_rate=0.2,
    steps=20,
)
for record in linear_trace[:5]:
    print(
        f"step={record.step:02d} parameters={record.parameters_before} "
        f"gradient={record.gradient} next_loss={record.loss_after:.8f}"
    )

In [ ]:
final_weight, final_bias = linear_trace[-1].parameters_after
final_linear_loss = linear_loss(xs, ys_with_bias, final_weight, final_bias)
print(f"final weight={final_weight:.8f}, bias={final_bias:.8f}, loss={final_linear_loss:.3e}")
assert abs(final_weight - 3.0) < 1.0e-10
assert abs(final_bias - 2.0) < 1.0e-4
assert final_linear_loss < 1.0e-8

## Code reading: parameter → prediction → loss → gradient → update

Read `one_parameter_step` in `missions/M19/gradient_core.py`. Point to the exact value emitted by each stage and consumed by the next. Then answer: if loss rises, which recorded values let you distinguish a wrong derivative from a wrong update sign?

In [ ]:
trace_names = ["parameter", "prediction", "loss", "gradient", "update"]
trace_values = [
    correct_step.parameters_before,
    correct_step.predictions,
    correct_step.loss_before,
    correct_step.gradient,
    correct_step.parameters_after,
]
for name, value in zip(trace_names, trace_values):
    print(f"{name:10s} -> {value}")

## No-AI gate

Close this notebook and complete `missions/M19/no_ai_gate.md` without AI-generated code or explanations. The gate requires a manual tiny gradient/update and a minimal repair of an unfamiliar faulty loop. Do not paste notebook values as your evidence; show every arithmetic step and the repaired four-step trace.

## Engineering handoff

Use `missions/M19/adr_prompt.md` to decide how V04 should verify analytic gradients before accepting multi-parameter updates. This is an architecture decision because a silent derivative or sign error contaminates later optimizer evidence.

## Explain the mechanism

In your own words, explain why a parameter changes, how its gradient was checked, what the next loss tells you, why the faulty sign failed, and how the scalar mechanism became a gradient vector. Stop before optimizer-family comparisons; those belong to M20.